# 05 — MCP Client-Server Communication

This notebook demonstrates the full client-server communication pattern.

## Learning Objectives

- Connect an MCP client to a server
- List tools, resources, and prompts
- Call tools and read resources
- Understand the message protocol

## Architecture Review

```
MCP Client                        MCP Server
  │                                  │
  ├── list_tools request ──────────► │
  │◄── list_tools result ──────────┤ │
  │                                  │
  ├── call_tool request  ──────────► │
  │   (name + arguments)             │ ──► executes function
  │◄── call_tool result  ──────────┤ │
  │                                  │
  ├── read_resource request ───────► │
  │   (URI)                          │ ──► fetches data
  │◄── read_resource result ───────┤ │
  │                                  │
  ├── get_prompt request  ─────────► │
  │   (name + args)                  │ ──► interpolates template
  │◄── get_prompt result  ─────────┤ │
```

In [ ]:
# The MCPClient class (from projects/mcp_document_client/client.py)
# This is the full implementation for reference

CLIENT_CODE = '''
import json
from typing import Optional, Any
from contextlib import AsyncExitStack
from mcp import ClientSession, StdioServerParameters, types
from mcp.client.stdio import stdio_client
from pydantic import AnyUrl


class MCPClient:
    def __init__(self, command: str, args: list[str], env: Optional[dict] = None):
        self._command = command
        self._args = args
        self._env = env
        self._session: Optional[ClientSession] = None
        self._exit_stack = AsyncExitStack()

    async def connect(self):
        server_params = StdioServerParameters(
            command=self._command, args=self._args, env=self._env
        )
        stdio_transport = await self._exit_stack.enter_async_context(
            stdio_client(server_params)
        )
        read_stream, write_stream = stdio_transport
        self._session = await self._exit_stack.enter_async_context(
            ClientSession(read_stream, write_stream)
        )
        await self._session.initialize()

    async def list_tools(self) -> list:
        result = await self._session.list_tools()
        return result.tools

    async def call_tool(self, tool_name: str, tool_input: dict):
        return await self._session.call_tool(tool_name, tool_input)

    async def read_resource(self, uri: str) -> Any:
        result = await self._session.read_resource(AnyUrl(uri))
        resource = result.contents[0]
        if resource.mimeType == "application/json":
            return json.loads(resource.text)
        return resource.text

    async def list_prompts(self) -> list:
        result = await self._session.list_prompts()
        return result.prompts

    async def get_prompt(self, name: str, args: dict):
        result = await self._session.get_prompt(name, args)
        return result.messages

    async def cleanup(self):
        await self._exit_stack.aclose()
        self._session = None

    async def __aenter__(self):
        await self.connect()
        return self

    async def __aexit__(self, *exc):
        await self.cleanup()
'''

print("MCPClient provides:")
print("  connect()        — establish connection")
print("  list_tools()     — discover available tools")
print("  call_tool()      — execute a tool")
print("  read_resource()  — fetch resource data")
print("  list_prompts()   — discover available prompts")
print("  get_prompt()     — fetch populated prompt")
print("  cleanup()        — close connection")

## Simulating Client-Server Interaction

Since we can't run an actual MCP server in a notebook cell, let's simulate the interaction:

In [ ]:
import json

# Simulated MCP Server state
docs = {
    "report.pdf": "The report details the state of a 20m condenser tower.",
    "plan.md": "The plan outlines the steps for implementation.",
    "spec.txt": "Technical requirements for the equipment.",
}

# Simulated server-side tool registry
server_tools = {
    "read_doc_contents": {
        "schema": {
            "name": "read_doc_contents",
            "description": "Read the contents of a document",
            "input_schema": {
                "type": "object",
                "properties": {"doc_id": {"type": "string", "description": "Document ID"}},
                "required": ["doc_id"],
            },
        },
        "handler": lambda args: docs.get(args["doc_id"], f"Not found: {args['doc_id']}"),
    },
    "edit_document": {
        "schema": {
            "name": "edit_document",
            "description": "Edit a document via find/replace",
            "input_schema": {
                "type": "object",
                "properties": {
                    "doc_id": {"type": "string"},
                    "old_str": {"type": "string"},
                    "new_str": {"type": "string"},
                },
                "required": ["doc_id", "old_str", "new_str"],
            },
        },
        "handler": lambda args: _edit(args),
    },
}

def _edit(args):
    doc_id, old_str, new_str = args["doc_id"], args["old_str"], args["new_str"]
    if doc_id not in docs:
        return f"Error: {doc_id} not found"
    docs[doc_id] = docs[doc_id].replace(old_str, new_str)
    return f"Updated {doc_id}"

In [ ]:
# Step 1: list_tools — Client discovers server capabilities
print("=== list_tools request ===")
tools = [t["schema"] for t in server_tools.values()]
print(f"Server returned {len(tools)} tools:")
for tool in tools:
    print(f"  - {tool['name']}: {tool['description']}")

In [ ]:
# Step 2: call_tool — Client executes a tool
print("=== call_tool request ===")
print("Tool: read_doc_contents")
print("Input: {\"doc_id\": \"report.pdf\"}")
print()

result = server_tools["read_doc_contents"]["handler"]({"doc_id": "report.pdf"})
print(f"=== call_tool result ===")
print(f"Result: {result}")

In [ ]:
# Step 3: Full agentic loop simulation
print("=== Full Agentic Loop ===")
print()

user_query = "What's in the report document?"
print(f"User: {user_query}")
print()

# Claude would receive: query + tool schemas
# Claude decides: use read_doc_contents(doc_id="report.pdf")
print("Claude decides: call read_doc_contents(doc_id='report.pdf')")
tool_result = server_tools["read_doc_contents"]["handler"]({"doc_id": "report.pdf"})
print(f"Tool result: {tool_result}")
print()

# Claude formulates response using the tool result
print("Claude: The report describes the current state of a 20m condenser tower.")

## Running the Real Client-Server

To test with actual MCP communication:

```bash
# Terminal 1: Test server with inspector
mcp dev projects/mcp_document_server/server.py

# Terminal 2: Run the CLI chatbot
python projects/cli_chatbot/main.py
```

## Exercise

1. Add a `list_resources` simulation to the code above
2. Simulate a `read_resource` call for `docs://documents`
3. Simulate a multi-tool conversation where Claude reads a doc, then edits it

In [ ]:
# Exercise: Multi-tool simulation
print("=== Multi-Tool Conversation ===")
print()
print("User: Rewrite the plan document to be more detailed")
print()

# Step 1: Claude reads the document
content = server_tools["read_doc_contents"]["handler"]({"doc_id": "plan.md"})
print(f"Claude calls: read_doc_contents('plan.md')")
print(f"Result: {content}")
print()

# Step 2: Claude edits the document
result = server_tools["edit_document"]["handler"]({
    "doc_id": "plan.md",
    "old_str": "The plan outlines the steps for implementation.",
    "new_str": "# Implementation Plan\n\n## Overview\nThis plan outlines the detailed steps for implementation.\n\n## Steps\n1. Requirements gathering\n2. Design phase\n3. Implementation\n4. Testing\n5. Deployment",
})
print(f"Claude calls: edit_document('plan.md', ...)")
print(f"Result: {result}")
print()
print(f"Updated document:")
print(docs["plan.md"])